#### Imports

In [3]:
import pandas as pd 
import numpy as np
import warnings
import matplotlib.pyplot as plt 

In [ ]:
warnings.filterwarnings("ignore")
pd.reset_option("all")
#pd.set_option('display.max_columns', None)
#pd.set_option("display.max_colwidth", None)

In [5]:
data = pd.read_excel(r"C:\Users\psxea2\Desktop\Amdaritelecomchurn\dataset.xlsx")


In [ ]:
data.head()

: 

: 

: 

: 

: 

: 

#### initial data exploration 

In [6]:
data.shape

(12483, 21)

In [7]:
data['PurchaseHistory'][0]

"[{'Product': 'Frozen Cocktail Mixes', 'Frequency': 8, 'Value': 884.43}, {'Product': 'Printer, Copier & Fax Machine Accessories', 'Frequency': 7, 'Value': 397.14}, {'Product': 'Hockey Stick Care', 'Frequency': 10, 'Value': 498.92}, {'Product': 'Guacamole', 'Frequency': 2, 'Value': 718.43}, {'Product': 'Mortisers', 'Frequency': 2, 'Value': 614.08}, {'Product': 'Rulers', 'Frequency': 6, 'Value': 221.68}, {'Product': 'Invitations', 'Frequency': 3, 'Value': 660.04}]"

In [8]:
import ast


In [9]:

normalised_df = pd.DataFrame({

    'CustomerID': data['CustomerID'],
    'purchaseHistory': data['PurchaseHistory']
})

normalised_df['purchaseHistory'] = normalised_df['purchaseHistory'].apply(ast.literal_eval)

In [10]:
normalised_df.dtypes

CustomerID          int64
purchaseHistory    object
dtype: object

In [11]:
normalised_df = pd.DataFrame(normalised_df)
normalised_df

,CustomerID,purchaseHistory
0,1001,"[{'Product': 'Frozen Cocktail Mixes', 'Frequen..."
1,1002,"[{'Product': 'Watercraft Polishes', 'Frequency..."
2,1003,"[{'Product': 'Vehicle Waxes, Polishes & Protec..."
3,1004,"[{'Product': 'Mouthwash', 'Frequency': 5, 'Val..."
4,1005,"[{'Product': 'Ice Cream Novelties', 'Frequency..."
...,...,...
12478,13479,"[{'Product': 'Ice Cream Novelties', 'Frequency..."
12479,13480,"[{'Product': 'Straight Pins', 'Frequency': 1, ..."
12480,13481,"[{'Product': 'Furisode Kimonos', 'Frequency': ..."
12481,13482,"[{'Product': 'Sequins & Glitter', 'Frequency':..."


In [12]:
normalised_df['purchaseHistory'][0]

[{'Product': 'Frozen Cocktail Mixes', 'Frequency': 8, 'Value': 884.43},
 {'Product': 'Printer, Copier & Fax Machine Accessories',
  'Frequency': 7,
  'Value': 397.14},
 {'Product': 'Hockey Stick Care', 'Frequency': 10, 'Value': 498.92},
 {'Product': 'Guacamole', 'Frequency': 2, 'Value': 718.43},
 {'Product': 'Mortisers', 'Frequency': 2, 'Value': 614.08},
 {'Product': 'Rulers', 'Frequency': 6, 'Value': 221.68},
 {'Product': 'Invitations', 'Frequency': 3, 'Value': 660.04}]

In [13]:
normalised_df.explode('purchaseHistory')

,CustomerID,purchaseHistory
0,1001,"{'Product': 'Frozen Cocktail Mixes', 'Frequenc..."
0,1001,"{'Product': 'Printer, Copier & Fax Machine Acc..."
0,1001,"{'Product': 'Hockey Stick Care', 'Frequency': ..."
0,1001,"{'Product': 'Guacamole', 'Frequency': 2, 'Valu..."
0,1001,"{'Product': 'Mortisers', 'Frequency': 2, 'Valu..."
...,...,...
12481,13482,"{'Product': 'Tractor Parts & Accessories', 'Fr..."
12481,13482,"{'Product': 'Video Cards & Adapters', 'Frequen..."
12481,13482,"{'Product': 'Clear Kerosene', 'Frequency': 3, ..."
12481,13482,"{'Product': 'Tripod Spreaders', 'Frequency': 8..."


In [14]:
normalised_df= normalised_df.explode('purchaseHistory')



In [15]:
normalised_df.columns

Index(['CustomerID', 'purchaseHistory'], dtype='object')

In [16]:
normalised_df

,CustomerID,purchaseHistory
0,1001,"{'Product': 'Frozen Cocktail Mixes', 'Frequenc..."
0,1001,"{'Product': 'Printer, Copier & Fax Machine Acc..."
0,1001,"{'Product': 'Hockey Stick Care', 'Frequency': ..."
0,1001,"{'Product': 'Guacamole', 'Frequency': 2, 'Valu..."
0,1001,"{'Product': 'Mortisers', 'Frequency': 2, 'Valu..."
...,...,...
12481,13482,"{'Product': 'Tractor Parts & Accessories', 'Fr..."
12481,13482,"{'Product': 'Video Cards & Adapters', 'Frequen..."
12481,13482,"{'Product': 'Clear Kerosene', 'Frequency': 3, ..."
12481,13482,"{'Product': 'Tripod Spreaders', 'Frequency': 8..."


In [17]:
normalised_df['purchaseHistory'].apply(pd.Series)

,Product,Frequency,Value
0,Frozen Cocktail Mixes,8,884.43
0,"Printer, Copier & Fax Machine Accessories",7,397.14
0,Hockey Stick Care,10,498.92
0,Guacamole,2,718.43
0,Mortisers,2,614.08
...,...,...,...
12481,Tractor Parts & Accessories,9,845.64
12481,Video Cards & Adapters,3,791.84
12481,Clear Kerosene,3,908.41
12481,Tripod Spreaders,8,238.25


In [18]:

pd.concat([
    normalised_df.drop(columns=['purchaseHistory']),
    normalised_df['purchaseHistory'].apply(pd.Series)
], axis=1)

,CustomerID,Product,Frequency,Value
0,1001,Frozen Cocktail Mixes,8,884.43
0,1001,"Printer, Copier & Fax Machine Accessories",7,397.14
0,1001,Hockey Stick Care,10,498.92
0,1001,Guacamole,2,718.43
0,1001,Mortisers,2,614.08
...,...,...,...,...
12481,13482,Tractor Parts & Accessories,9,845.64
12481,13482,Video Cards & Adapters,3,791.84
12481,13482,Clear Kerosene,3,908.41
12481,13482,Tripod Spreaders,8,238.25


### create a function to automatically normalise the  respective columns in the data

In [19]:
def normalize (df: pd.DataFrame,column:str, fk:str) -> pd.DataFrame:
    """
    - a function to extract respective columns and transform them as a DataFrame
    - respective columns will have a unique primary key and single value in each coulmn
    - the function outputs a dataframe with normalised columns

    """

    normalized_df = pd.DataFrame({
    fk: df[fk],
    column : df[column]
    })

    normalized_df[column]=normalized_df[column].apply(ast.literal_eval)


    if isinstance(normalized_df[column][0], list):
        normalized_df = normalized_df.explode(column)


    normalized_df = pd.concat([
    normalized_df.drop(columns = [column]),
    normalized_df[column].apply(pd.Series)
    ], axis=1)


    return normalized_df





In [21]:
data.columns

Index(['CustomerID', 'Name', 'Age', 'Gender', 'Location', 'Email', 'Phone',
       'Address', 'Segment', 'PurchaseHistory', 'SubscriptionDetails',
       'ServiceInteractions', 'PaymentHistory', 'WebsiteUsage',
       'ClickstreamData', 'EngagementMetrics', 'Feedback',
       'MarketingCommunication', 'NPS', 'ChurnLabel', 'Timestamp'],
      dtype='object')

### test the normalisation function

In [22]:
normalize(data, 'PurchaseHistory', 'CustomerID')

,CustomerID,Product,Frequency,Value
0,1001,Frozen Cocktail Mixes,8,884.43
0,1001,"Printer, Copier & Fax Machine Accessories",7,397.14
0,1001,Hockey Stick Care,10,498.92
0,1001,Guacamole,2,718.43
0,1001,Mortisers,2,614.08
...,...,...,...,...
12481,13482,Tractor Parts & Accessories,9,845.64
12481,13482,Video Cards & Adapters,3,791.84
12481,13482,Clear Kerosene,3,908.41
12481,13482,Tripod Spreaders,8,238.25


In [23]:
#data['WebsiteUsage']

In [24]:
#check which columns need normalisation
data.columns.to_list()[9:18]

['PurchaseHistory',
 'SubscriptionDetails',
 'ServiceInteractions',
 'PaymentHistory',
 'WebsiteUsage',
 'ClickstreamData',
 'EngagementMetrics',
 'Feedback',
 'MarketingCommunication']

In [25]:
#use indices to slice the dataframe and get the column names in a list
data.iloc[:,9:18].columns.to_list()

['PurchaseHistory',
 'SubscriptionDetails',
 'ServiceInteractions',
 'PaymentHistory',
 'WebsiteUsage',
 'ClickstreamData',
 'EngagementMetrics',
 'Feedback',
 'MarketingCommunication']

In [26]:
#create a list of columns to normalize and apply the normalize function to each of those columns
cols_to_normalize = data.iloc[:,9:18].columns.to_list()

#create an empty dictionary to store the normalized dataframes:normalized_cols

normalized_cols = {} #The dictionary `normalized_cols` contains items where the keys are column names (strings), 
# and the values are pandas DataFrames. So, it is a dictionary of type `Dict[str, pd.DataFrame]`.


for col in cols_to_normalize:
    normalized_cols[col] = normalize(data, col, 'CustomerID')


In [27]:
# view the results of one of the  normalized columns: purchaseHistory
normalized_cols['PurchaseHistory']

,CustomerID,Product,Frequency,Value
0,1001,Frozen Cocktail Mixes,8,884.43
0,1001,"Printer, Copier & Fax Machine Accessories",7,397.14
0,1001,Hockey Stick Care,10,498.92
0,1001,Guacamole,2,718.43
0,1001,Mortisers,2,614.08
...,...,...,...,...
12481,13482,Tractor Parts & Accessories,9,845.64
12481,13482,Video Cards & Adapters,3,791.84
12481,13482,Clear Kerosene,3,908.41
12481,13482,Tripod Spreaders,8,238.25


In [28]:
normalized_cols['WebsiteUsage']

,CustomerID,PageViews,TimeSpent(minutes)
0,1001,49,15
1,1002,100,9
2,1003,1,97
3,1004,25,31
4,1005,77,51
...,...,...,...
12478,13479,70,57
12479,13480,71,66
12480,13481,96,1
12481,13482,63,2


In [29]:
normalized_cols['ClickstreamData']

,CustomerID,Action,Page,Timestamp
0,1001,Add to Cart,register,2020-09-13 17:06:44
0,1001,Search,login,2022-03-30 14:51:52
0,1001,Click,about,2019-11-10 05:48:48
0,1001,Add to Cart,terms,2019-05-15 10:17:44
0,1001,Add to Cart,author,2022-07-14 03:40:53
...,...,...,...,...
12482,13483,Search,terms,2020-05-25 09:12:57
12482,13483,Search,faq,2022-10-22 16:58:27
12482,13483,Search,index,2019-12-02 01:00:43
12482,13483,Add to Cart,author,2019-07-13 11:12:37


In [30]:
normalized_cols['EngagementMetrics']

,CustomerID,Logins,Frequency
0,1001,19,Weekly
1,1002,9,Weekly
2,1003,19,Monthly
3,1004,4,Daily
4,1005,12,Weekly
...,...,...,...
12478,13479,22,Daily
12479,13480,25,Weekly
12480,13481,9,Monthly
12481,13482,2,Monthly


In [31]:
normalized_cols['Feedback']

,CustomerID,Rating,Comment
0,1001,1,I move baby go small big. Office institution s...
1,1002,2,Wish what bag cut life. Statement might opport...
2,1003,4,Some Democrat guess but short. Whether behind ...
3,1004,1,Yard feel never miss ask billion Congress. Fly...
4,1005,3,Ten determine unit interview challenge stock. ...
...,...,...,...
12478,13479,2,Light appear fight lawyer where star.
12479,13480,3,Yet very girl history. Thing late dream you re...
12480,13481,5,Offer particularly single degree seem sound. S...
12481,13482,5,Rest something concern likely movie. Foot in i...


In [32]:
normalized_cols['MarketingCommunication']

,CustomerID,Email_Sent,Email_Opened,Email_Clicked
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
...,...,...,...,...
12482,13483,2021-02-08,2021-09-18,2022-04-11
12482,13483,2021-02-08,2021-09-18,2022-04-11
12482,13483,2021-02-08,2021-09-18,2022-04-11
12482,13483,2021-02-08,2021-09-18,2022-04-11


In [33]:
normalized_cols['PaymentHistory']

,CustomerID,Method,Late_Payments
0,1001,Credit Card,5
0,1001,PayPal,11
0,1001,Bank Transfer,24
1,1002,Credit Card,3
1,1002,PayPal,3
...,...,...,...
12481,13482,PayPal,3
12481,13482,Bank Transfer,34
12482,13483,Credit Card,11
12482,13483,PayPal,8


In [34]:
normalized_cols['ServiceInteractions']

,CustomerID,Type,Date
0,1001,Call,2019-09-26
0,1001,Chat,2021-07-25
0,1001,Email,2020-04-13
0,1001,Chat,2020-11-15
1,1002,Call,2020-01-05
...,...,...,...
12482,13483,Call,2022-04-06
12482,13483,Call,2021-01-31
12482,13483,Chat,2021-11-12
12482,13483,Call,2022-06-10


In [35]:
normalized_cols['SubscriptionDetails']

,CustomerID,Plan,Start_Date,End_Date
0,1001,Express,2020-06-08,2022-10-27
1,1002,Pro,2021-07-21,2022-05-07
2,1003,Essential,2019-10-05,2020-08-19
3,1004,Smart,2020-01-14,2022-03-27
4,1005,Basic,2021-04-08,2022-11-09
...,...,...,...,...
12478,13479,Essential,2019-06-15,2021-06-29
12479,13480,Flex,2022-12-10,2022-12-28
12480,13481,Deluxe,2021-07-04,2021-07-24
12481,13482,Gold,2020-07-21,2021-11-17


In [36]:
normalized_cols['EngagementMetrics']

,CustomerID,Logins,Frequency
0,1001,19,Weekly
1,1002,9,Weekly
2,1003,19,Monthly
3,1004,4,Daily
4,1005,12,Weekly
...,...,...,...
12478,13479,22,Daily
12479,13480,25,Weekly
12480,13481,9,Monthly
12481,13482,2,Monthly


In [37]:
# Check if any of the normalized columns have multiple rows for the same CustomerID
# This will help identify columns that may need further normalization or investigation.

duplicate_id_status = {
# Dictionary comprehension to create a dictionary: multiple_rows_status- 
# where keys are column names and values are either True or False 
# indicating if there are multiple rows for the same CustomerID in that column's DataFrame.

    col: normalized_cols[col]['CustomerID'].duplicated(keep=False).any()
    for col in normalized_cols
} 

# List columns where the same CustomerID has multiple rows
cols_with_duplicate_idrows = [col for col, has_multiple in duplicate_id_status.items() if has_multiple]

print("Columns in normalized_cols where the same CustomerID appears in multiple rows:")
for col in cols_with_duplicate_idrows:
    print(f"- {col}")


print("\n")

print("Columns in normalized_cols where each row has a unique customerID:")
for col in normalized_cols:
    if not duplicate_id_status[col]:
        print(f"- {col}")  


duplicate_id_status

Columns in normalized_cols where the same CustomerID appears in multiple rows:
- PurchaseHistory
- ServiceInteractions
- PaymentHistory
- ClickstreamData
- MarketingCommunication


Columns in normalized_cols where each row has a unique customerID:
- SubscriptionDetails
- WebsiteUsage
- EngagementMetrics
- Feedback


{'PurchaseHistory': np.True_,
 'SubscriptionDetails': np.False_,
 'ServiceInteractions': np.True_,
 'PaymentHistory': np.True_,
 'WebsiteUsage': np.False_,
 'ClickstreamData': np.True_,
 'EngagementMetrics': np.False_,
 'Feedback': np.False_,
 'MarketingCommunication': np.True_}

In [38]:
"""
Alternative approach to check for multiple rows per CustomerID
 This method compares the total number of rows to the number of unique CustomerIDs.
 If the total number of rows is greater than the number of unique CustomerIDs, 
 it indicates that there are multiple rows for some CustomerIDs in that particular dataframe.
 
"""

nonunique_idstatus = {
    col: len(normalized_cols[col]) > normalized_cols[col]['CustomerID'].nunique()
    for col in normalized_cols
}


# List the columns that have a dataframe  where the same CustomerID appears in multiple rows 
cols_with_nonunique_idrows = [col for col, has_multiple in nonunique_idstatus.items() if has_multiple]

print("Columns in normalized_cols where their dataframe has the same CustomerID appearing in multiple rows:")
for col in cols_with_nonunique_idrows:
    print(f"- {col}")

print("\n")  # Print a newline for better readability

cols_with_unique_idrows = [col for col in normalized_cols if not nonunique_idstatus[col]]
#print(cols_with_unique_idrows)

print("Columns in normalized_cols where each row in their dataframe has a unique customerID:")
for col in normalized_cols:
    if not nonunique_idstatus[col]:
        print(f"- {col}")  


nonunique_idstatus

Columns in normalized_cols where their dataframe has the same CustomerID appearing in multiple rows:
- PurchaseHistory
- ServiceInteractions
- PaymentHistory
- ClickstreamData
- MarketingCommunication


Columns in normalized_cols where each row in their dataframe has a unique customerID:
- SubscriptionDetails
- WebsiteUsage
- EngagementMetrics
- Feedback


{'PurchaseHistory': True,
 'SubscriptionDetails': False,
 'ServiceInteractions': True,
 'PaymentHistory': True,
 'WebsiteUsage': False,
 'ClickstreamData': True,
 'EngagementMetrics': False,
 'Feedback': False,
 'MarketingCommunication': True}

In [39]:
normalized_cols.keys() 

dict_keys(['PurchaseHistory', 'SubscriptionDetails', 'ServiceInteractions', 'PaymentHistory', 'WebsiteUsage', 'ClickstreamData', 'EngagementMetrics', 'Feedback', 'MarketingCommunication'])

### further feature engineering on those normalized columns where their dataframes have the customerid appearing in multiple rows

In [40]:
cols_with_nonunique_idrows

['PurchaseHistory',
 'ServiceInteractions',
 'PaymentHistory',
 'ClickstreamData',
 'MarketingCommunication']

#### PurchaseHistory

In [82]:
normalized_cols['PurchaseHistory'].head()

,CustomerID,Product,Frequency,Value
0,1001,Frozen Cocktail Mixes,8,884.43
0,1001,"Printer, Copier & Fax Machine Accessories",7,397.14
0,1001,Hockey Stick Care,10,498.92
0,1001,Guacamole,2,718.43
0,1001,Mortisers,2,614.08


In [42]:
import textwrap
# engineer features from the PurchaseHistory dataframe by aggregating data for each CustomerID
# Create a new column called total_purchases by applying 'sum' on the  Frequency column
# create a new column called total_value by applying 'sum' on the Value column
# create a new column called productslist by applying a lambda function that joins unique product names with commas
# group by CustomerID to ensure one row per customer

purchaseHistory = normalized_cols['PurchaseHistory'].groupby('CustomerID').agg(
    TotalPurchaseFrequency=('Frequency', 'sum'),
    TotalPurchaseValue= ('Value', 'sum'),
    ProductList=('Product', 
                  lambda x: " ".join(
                    textwrap.wrap(', '.join(sorted(set(x))), 
                                width=40)
                  )    
)).reset_index()

purchaseHistory.head()


,CustomerID,TotalPurchaseFrequency,TotalPurchaseValue,ProductList
0,1001,38,3994.72,"Frozen Cocktail Mixes, Guacamole, Hockey Stick..."
1,1002,4,2844.35,"Baby Protective Wear, Footbags, Watercraft Pol..."
2,1003,14,1866.52,"Fudge, Pipe Caps & Plugs, Vehicle Waxes, Polis..."
3,1004,28,1378.64,"Crêpe & Blini Pans, Electrical Switches, Mouth..."
4,1005,39,2425.05,"Audiobooks, Fire Extinguisher & Equipment Stor..."


In [ ]:


def wrap_products(s: str, max_items: int = 4) -> str:
    """
    Args:
        s (str): A comma-separated string of product names.
        max_items (int): Maximum number of products to include per line. Default = 4.
    
    Returns:
        str: A multi-line string where each line contains up to `max_items` product names,
             separated by commas.
    """
    products = [p.strip() for p in s.split(",")]   # split the long list into product names
    chunks = [
        ",".join(products[i:i+max_items])         # group products into lines
        for i in range(0, len(products), max_items)
    ]

    return "\n".join(chunks)   # use <br> for HTML line breaks instead of \n


# Apply to your dataframe 
purchaseHistory["productslist"] = purchaseHistory["productslist"].apply(lambda x: wrap_products(x, max_items=3))


#purchaseHistory.head(10).style.format({"productslist": lambda v: v}, escape="html")


##from IPython.display import display

#style=purchaseHistory.head(5).style.set_properties(
    #subset=["productslist"], **{"white-space": "pre-wrap"}
#)

#display(style)

#### PaymentHistory

In [43]:
normalized_cols['PaymentHistory'].head()


,CustomerID,Method,Late_Payments
0,1001,Credit Card,5
0,1001,PayPal,11
0,1001,Bank Transfer,24
1,1002,Credit Card,3
1,1002,PayPal,3


In [44]:
normalized_cols['PaymentHistory']['Method'].value_counts()

Method
Credit Card      12483
PayPal           12483
Bank Transfer    12483
Name: count, dtype: int64

In [45]:
import textwrap
# engineer features from the PaymentHistory dataframe by aggregating data for each CustomerID
# Create a new column called NumPaymentMethod which is the count of unique values in Method column
# Create a new column called AvgLatePayment which is the mean of Late_Payments column


paymentHistory = normalized_cols['PaymentHistory'].groupby('CustomerID').agg(
    
    PaymentTypes=('Method', lambda x: ",".join(sorted(set(x)))),
    NumPaymentMethod=('Method', 'nunique'),
    #FreqBankPaymnt=('Method', lambda x: (x=='Bank').sum()),
    #FreqCreditCardPmt=('Method', lambda x: (x=='Card').sum()),
    #FreqPayPalPmt=('Method', lambda x: (x=='PayPal').sum()),
    AvgLatePayment=('Late_Payments', 'mean'),

).reset_index()

paymentHistory.head()

,CustomerID,PaymentTypes,NumPaymentMethod,AvgLatePayment
0,1001,"Bank Transfer,Credit Card,PayPal",3,13.333333
1,1002,"Bank Transfer,Credit Card,PayPal",3,3.333333
2,1003,"Bank Transfer,Credit Card,PayPal",3,2.666667
3,1004,"Bank Transfer,Credit Card,PayPal",3,26.333333
4,1005,"Bank Transfer,Credit Card,PayPal",3,0.666667


#### ServiceInteractions

In [46]:
normalized_cols['ServiceInteractions'].head()

,CustomerID,Type,Date
0,1001,Call,2019-09-26
0,1001,Chat,2021-07-25
0,1001,Email,2020-04-13
0,1001,Chat,2020-11-15
1,1002,Call,2020-01-05


In [47]:
normalized_cols['ServiceInteractions']['Type'].value_counts()

Type
Email    85359
Chat     84630
Call     84264
Name: count, dtype: int64

In [48]:
normalized_cols['ServiceInteractions']['Type'].isnull()

0        False
0        False
0        False
0        False
1        False
         ...  
12482    False
12482    False
12482    False
12482    False
12482    False
Name: Type, Length: 254253, dtype: bool

In [ ]:
#engineer features from the ServiceInteractions dataframe by aggregating data for each CustomerID
# Create a new column called ServiceInteractionType which is the count of unique values in Type column
#
normalized_cols['ServiceInteractions']['Date'] = pd.to_datetime(
    normalized_cols['ServiceInteractions']['Date'], errors='coerce'
)


ServiceInteractions = normalized_cols['ServiceInteractions'].groupby('CustomerID').agg(
    TotalInteractionType=('Type', lambda x: ','.join(sorted(set(x)))),
    #UniqueInteractions=('Type', 'nunique'),  
    #MostFreq_SIT=('Type', lambda x: x.mode()[0] if not x.mode().empty else np.nan), 
    num_calls=('Type', lambda x: (x == 'Call').sum()),
    num_emails=('Type', lambda x: (x == 'Email').sum()),
    num_chats=('Type', lambda x: (x == 'Chat').sum()),
    FirstInteractionDate=('Date', 'min'),
    LastInteractionDate=('Date', 'max'),
    InteractionDuration_days=('Date', lambda x: (x.max() - x.min()).days + 1 if len(x) > 1 else 1)
).reset_index()

ServiceInteractions.head()

NameError: name 'pd' is not defined

In [65]:
# For each customer, get the service type used on their first interaction
first_service_type = (
    normalized_cols['ServiceInteractions']
    .sort_values(['CustomerID', 'Date'])
    .groupby('CustomerID')
    .first()
    .reset_index()[['CustomerID', 'Type']]
    .rename(columns={'Type': 'FirstServiceType'})
)

first_service_type.head()

,CustomerID,FirstServiceType
0,1001,Call
1,1002,Call
2,1003,Email
3,1004,Email
4,1005,Call


In [66]:
# For each customer, get the service type used on their last interaction
last_service_type = (
    normalized_cols['ServiceInteractions']
    .sort_values(['CustomerID', 'Date'])
    .groupby('CustomerID')
    .last()
    .reset_index()[['CustomerID', 'Type']]
    .rename(columns={'Type': 'LastServiceType'})
)

last_service_type.head()

,CustomerID,LastServiceType
0,1001,Chat
1,1002,Email
2,1003,Call
3,1004,Email
4,1005,Call


In [67]:
# Add first and last service interaction type for each customer, avoiding duplicate columns and ordering as requested

# Merge with first and last service type and date, then reorder columns
ServiceInteractions = (
    ServiceInteractions
    .merge(first_service_type, on='CustomerID', how='left')
    .merge(last_service_type, on='CustomerID', how='left')
    .rename(columns={
        'FirstServiceType': 'FirstInteractionType',
        'LastServiceType': 'LastInteractionType'
    })
)

# Reorder columns as specified
cols = [
    'CustomerID', 'TotalInteractionType', 'num_calls', 'num_emails', 'num_chats',
    'FirstInteractionDate', 'FirstInteractionType',
    'LastInteractionDate', 'LastInteractionType', 'InteractionDuration_days'
]
serviceInteractions = ServiceInteractions[cols]

serviceInteractions.head()

,CustomerID,TotalInteractionType,num_calls,num_emails,num_chats,FirstInteractionDate,FirstInteractionType,LastInteractionDate,LastInteractionType,InteractionDuration_days
0,1001,"Call,Chat,Email",1,1,2,2019-09-26,Call,2021-07-25,Chat,669
1,1002,"Call,Chat,Email",5,10,4,2019-01-12,Call,2022-12-13,Email,1432
2,1003,"Call,Chat,Email",1,1,1,2019-10-09,Email,2022-01-04,Call,819
3,1004,"Call,Chat,Email",17,18,24,2019-01-03,Email,2022-11-10,Email,1408
4,1005,"Call,Chat,Email",4,5,1,2019-04-10,Call,2022-12-19,Call,1350


In [68]:
cols_with_nonunique_idrows

['PurchaseHistory',
 'ServiceInteractions',
 'PaymentHistory',
 'ClickstreamData',
 'MarketingCommunication']

#### ClickstreamData

In [69]:
normalized_cols['ClickstreamData']['Action'].value_counts()

Action
Click          106697
Search         106483
Add to Cart    106436
Name: count, dtype: int64

In [70]:
normalized_cols['ClickstreamData']['Page'].value_counts()

Page
author      23002
about       22968
homepage    22938
main        22856
category    22839
post        22834
index       22829
register    22821
privacy     22814
login       22814
terms       22796
home        22795
faq         22715
search      22595
Name: count, dtype: int64

In [71]:
ClickstreamData= normalized_cols['ClickstreamData']

In [72]:
import textwrap

normalized_cols['ClickstreamData']['Timestamp'] = pd.to_datetime(
    normalized_cols['ClickstreamData']['Timestamp'], errors='coerce'
)

# End of dataset
ClickstreamData_end = normalized_cols['ClickstreamData']['Timestamp'].max()


ClickstreamData = normalized_cols['ClickstreamData'].groupby('CustomerID').agg(

    #Action=('Action', lambda x: ','.join((sorted(set(x))))),
    Action_count=('Action', 'count'),
    #uniqueActions=('Action', 'nunique'),   
    #AddToCartFreq=('Action', lambda x: (x == 'Add to Cart').sum()),
    #SearchFreq=('Action', lambda x: (x == 'Search').sum()),
    #clickFreq=('Action', lambda x: (x == 'Click').sum()),
    FirstActionTime = ('Timestamp', 'min'),
    LastActionTime = ('Timestamp', 'max'),

    AvgTimeBetweenActions_secs=('Timestamp', lambda x: (
            np.mean(np.diff(np.sort(x.values)).astype('timedelta64[s]').astype(int))
            if len(x) > 1 else 0)),

    TotalDaysActive = ('Timestamp', lambda x: x.dt.date.nunique()),
    MostCommonAction= ('Action', lambda x: ", ".join(x.value_counts().index[x.value_counts() == x.value_counts().max()])
                    ),
    LeastFrequentAction= ('Action', 
                lambda x: ", ".join(x.value_counts().index[x.value_counts() == x.value_counts().min()])
                ),
    ActivityDuration_days = ('Timestamp', lambda x: (x.max() - x.min()).days + 1 if len(x) > 1 else 1),
    ActionsPerDay=('Timestamp', lambda x: len(x) / x.dt.date.nunique()),
    most_recent_action_date = ('Timestamp', 'max'),
    TotalPageVisits=('Page', 'count'),


    unique_pages=('Page', 'nunique'),
    #PageTitles=('Page', lambda x: ",".join(sorted(set(x)))),
    #PageTitles=(
            #"Page",
           # lambda x: " ".join(
            #    textwrap.wrap(",".join(sorted(set(x))), width=30))
        #),
    #MostVisitedPg=('Page', lambda x: x.value_counts().idxmax() if not x.empty else np.nan),
    #LeastVisitedPg=('Page', lambda x: x.value_counts().idxmin() if not x.empty else np.nan)
    
    ).reset_index()



ClickstreamData.head()

,CustomerID,Action_count,FirstActionTime,LastActionTime,AvgTimeBetweenActions_secs,TotalDaysActive,MostCommonAction,LeastFrequentAction,ActivityDuration_days,ActionsPerDay,most_recent_action_date,TotalPageVisits,unique_pages
0,1001,24,2019-01-13 08:39:42,2022-11-07 02:24:31,5.235613e+06,23,Search,Click,1394,1.043478,2022-11-07 02:24:31,24,13
1,1002,24,2019-04-17 04:22:41,2022-12-05 00:27:08,4.988046e+06,24,Click,Search,1328,1.000000,2022-12-05 00:27:08,24,13
2,1003,12,2019-03-09 05:16:38,2022-11-02 15:59:43,1.048147e+07,12,Search,Add to Cart,1335,1.000000,2022-11-02 15:59:43,12,7
3,1004,47,2019-02-08 13:10:41,2022-12-08 18:10:18,2.628078e+06,46,"Click, Search",Add to Cart,1400,1.021739,2022-12-08 18:10:18,47,14
4,1005,30,2019-01-08 20:16:44,2022-12-30 23:37:00,4.326373e+06,30,Add to Cart,Search,1453,1.000000,2022-12-30 23:37:00,30,12


In [ ]:
ClickstreamData.columns

In [73]:
first_action_type = (
    normalized_cols['ClickstreamData']
    .sort_values(['CustomerID', 'Timestamp'])
    .groupby('CustomerID')
    .first()
    .reset_index()[['CustomerID', 'Action']]
    .rename(columns={'Action': 'FirstActionType'})
)

In [ ]:
first_action_type.head()

In [74]:
last_action_type = (
    normalized_cols['ClickstreamData']
    .sort_values(['CustomerID', 'Timestamp'])
    .groupby('CustomerID')
    .last()
    .reset_index()[['CustomerID', 'Action']]
    .rename(columns={'Action': 'LastActionType'})
)

In [ ]:
last_action_type.head()

In [ ]:
ClickstreamData.columns

In [75]:
# Merge with first and last action type and date, then reorder columns
ClickstreamData = (
    ClickstreamData
    .merge(first_action_type, on='CustomerID', how='left')
    .merge(last_action_type, on='CustomerID', how='left')
    
)

#Reorder columns as specified

#cols = [
 #'CustomerID',
 # Actions
 #'Total_Actions', 'uniqueActions', 'AddToCartFreq', 'SearchFreq', 'clickFreq',
 #'MostCommonAction', 'LeastFrequentAction', 'Action',
 # Time
# 'FirstActionTime', 'LastActionTime', 'ActivityDuration_days', 
 #'TotalDaysActive', 'ActionsPerDay',
 # Pages
 #'TotalPageVisits', 'uniquePgs', 'MostVisitedPg', 'LeastVisitedPg', 'PageTitles'
#]
#ClickstreamData = ClickstreamData[cols]

ClickstreamData.head()

,CustomerID,Action_count,FirstActionTime,LastActionTime,AvgTimeBetweenActions_secs,TotalDaysActive,MostCommonAction,LeastFrequentAction,ActivityDuration_days,ActionsPerDay,most_recent_action_date,TotalPageVisits,unique_pages,FirstActionType,LastActionType
0,1001,24,2019-01-13 08:39:42,2022-11-07 02:24:31,5.235613e+06,23,Search,Click,1394,1.043478,2022-11-07 02:24:31,24,13,Search,Search
1,1002,24,2019-04-17 04:22:41,2022-12-05 00:27:08,4.988046e+06,24,Click,Search,1328,1.000000,2022-12-05 00:27:08,24,13,Click,Add to Cart
2,1003,12,2019-03-09 05:16:38,2022-11-02 15:59:43,1.048147e+07,12,Search,Add to Cart,1335,1.000000,2022-11-02 15:59:43,12,7,Click,Search
3,1004,47,2019-02-08 13:10:41,2022-12-08 18:10:18,2.628078e+06,46,"Click, Search",Add to Cart,1400,1.021739,2022-12-08 18:10:18,47,14,Add to Cart,Search
4,1005,30,2019-01-08 20:16:44,2022-12-30 23:37:00,4.326373e+06,30,Add to Cart,Search,1453,1.000000,2022-12-30 23:37:00,30,12,Add to Cart,Search


In [76]:

firstpagevisits = ( 
    normalized_cols['ClickstreamData']
    .sort_values(['CustomerID', 'Timestamp'])
    .groupby('CustomerID')
    .first()                                    
    .reset_index()[['CustomerID', 'Page']]
    .rename(columns={'Page': 'FirstPageVisited'})
)

lastpagevisits = ( 
    normalized_cols['ClickstreamData']
    .sort_values(['CustomerID', 'Timestamp'])
    .groupby('CustomerID')
    .last()                                      
    .reset_index()[['CustomerID', 'Page']]
    .rename(columns={'Page': 'LastPageVisited'})
)





In [77]:
ClickstreamData = (
    ClickstreamData
    .merge(firstpagevisits, on='CustomerID', how='left')
    .merge(lastpagevisits, on='CustomerID', how='left')
)

In [78]:
ClickstreamData.head()

,CustomerID,Action_count,FirstActionTime,LastActionTime,AvgTimeBetweenActions_secs,TotalDaysActive,MostCommonAction,LeastFrequentAction,ActivityDuration_days,ActionsPerDay,most_recent_action_date,TotalPageVisits,unique_pages,FirstActionType,LastActionType,FirstPageVisited,LastPageVisited
0,1001,24,2019-01-13 08:39:42,2022-11-07 02:24:31,5.235613e+06,23,Search,Click,1394,1.043478,2022-11-07 02:24:31,24,13,Search,Search,main,author
1,1002,24,2019-04-17 04:22:41,2022-12-05 00:27:08,4.988046e+06,24,Click,Search,1328,1.000000,2022-12-05 00:27:08,24,13,Click,Add to Cart,homepage,faq
2,1003,12,2019-03-09 05:16:38,2022-11-02 15:59:43,1.048147e+07,12,Search,Add to Cart,1335,1.000000,2022-11-02 15:59:43,12,7,Click,Search,author,about
3,1004,47,2019-02-08 13:10:41,2022-12-08 18:10:18,2.628078e+06,46,"Click, Search",Add to Cart,1400,1.021739,2022-12-08 18:10:18,47,14,Add to Cart,Search,category,about
4,1005,30,2019-01-08 20:16:44,2022-12-30 23:37:00,4.326373e+06,30,Add to Cart,Search,1453,1.000000,2022-12-30 23:37:00,30,12,Add to Cart,Search,register,post


In [79]:
    # --- Add recency features ---: how long ago the customer interacted with the website
ClickstreamData['Recency_days'] = (ClickstreamData_end - ClickstreamData['LastActionTime']).dt.days

ClickstreamData['InactivityFlag'] = (ClickstreamData['Recency_days'] > 30).astype(int)



# Actions in last 30 days

# Define the cutoff date for "last 30 days"
# Example: if dataset_end = 2025-09-20, cutoff = 2025-08-21
last30_cutoff = ClickstreamData_end - pd.Timedelta(days=30)


# Count how many actions each customer performed in the last 30 days
actions_last30 = (
    normalized_cols['ClickstreamData']
    # 1. Filter rows where Timestamp >= cutoff (only keep last 30 days)
    .loc[normalized_cols['ClickstreamData']['Timestamp'] >= last30_cutoff]
    # 2. Group by CustomerID (each customer's activity)
    .groupby('CustomerID')
    # 3. Count number of rows (actions/events) per customer
    .size()
     # 4. Rename the result to "ActionsLast30Days"
    .rename('ActionsLast30Days')
    # 5. Reset index so it's a proper DataFrame with two columns:
    #    CustomerID | ActionsLast30Days
    .reset_index()
)
ClickstreamData = ClickstreamData.merge(actions_last30, on='CustomerID', how='left').fillna({'ActionsLast30Days': 0})

# Active in last week

# Define the cutoff date for "last 7 days"
# Example: if dataset_end = 2025-09-20, cutoff = 2025-09-13
last7_cutoff = ClickstreamData_end - pd.Timedelta(days=7)


# Get all customers who had at least one action in the last 7 days
# 1. Filter rows where Timestamp >= cutoff
# 2. Select only the CustomerID column
# 3. Take the unique IDs (so each customer appears once)
# Result = a NumPy array of active customer IDs
active_last7_customers = normalized_cols['ClickstreamData'].loc[
    normalized_cols['ClickstreamData']['Timestamp'] >= last7_cutoff, 'CustomerID'
].unique()


# Create a new binary flag column in ClickstreamData:
#  - 1 if this customer's ID is in the active_last7_customers list
#  - 0 otherwise
ClickstreamData['ActiveInLastWeek'] = ClickstreamData['CustomerID'].isin(active_last7_customers).astype(int)


ClickstreamData.head()

,CustomerID,Action_count,FirstActionTime,LastActionTime,AvgTimeBetweenActions_secs,TotalDaysActive,MostCommonAction,LeastFrequentAction,ActivityDuration_days,ActionsPerDay,...,TotalPageVisits,unique_pages,FirstActionType,LastActionType,FirstPageVisited,LastPageVisited,Recency_days,InactivityFlag,ActionsLast30Days,ActiveInLastWeek
0,1001,24,2019-01-13 08:39:42,2022-11-07 02:24:31,5.235613e+06,23,Search,Click,1394,1.043478,...,24,13,Search,Search,main,author,53,1,0.0,0
1,1002,24,2019-04-17 04:22:41,2022-12-05 00:27:08,4.988046e+06,24,Click,Search,1328,1.000000,...,24,13,Click,Add to Cart,homepage,faq,25,0,1.0,0
2,1003,12,2019-03-09 05:16:38,2022-11-02 15:59:43,1.048147e+07,12,Search,Add to Cart,1335,1.000000,...,12,7,Click,Search,author,about,58,1,0.0,0
3,1004,47,2019-02-08 13:10:41,2022-12-08 18:10:18,2.628078e+06,46,"Click, Search",Add to Cart,1400,1.021739,...,47,14,Add to Cart,Search,category,about,22,0,1.0,0
4,1005,30,2019-01-08 20:16:44,2022-12-30 23:37:00,4.326373e+06,30,Add to Cart,Search,1453,1.000000,...,30,12,Add to Cart,Search,register,post,0,0,3.0,1


In [ ]:
# view raw data columns
data.columns

Index(['CustomerID', 'Name', 'Age', 'Gender', 'Location', 'Email', 'Phone',
       'Address', 'Segment', 'PurchaseHistory', 'SubscriptionDetails',
       'ServiceInteractions', 'PaymentHistory', 'WebsiteUsage',
       'ClickstreamData', 'EngagementMetrics', 'Feedback',
       'MarketingCommunication', 'NPS', 'ChurnLabel', 'Timestamp'],
      dtype='object')

#### MarketingCommunications

In [81]:
cols_with_nonunique_idrows

['PurchaseHistory',
 'ServiceInteractions',
 'PaymentHistory',
 'ClickstreamData',
 'MarketingCommunication']

In [85]:
df_mc = normalized_cols['MarketingCommunication']

df_mc.head()

,CustomerID,Email_Sent,Email_Opened,Email_Clicked
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27
0,1001,2019-10-17,2022-01-12,2022-11-27


In [87]:
# Make sure columns are datetime
for col in ['Email_Sent', 'Email_Opened', 'Email_Clicked']:
    normalized_cols['MarketingCommunication'][col] = pd.to_datetime(
        normalized_cols['MarketingCommunication'][col],
        errors='coerce'
    )


# Calculate per-row differences in days
df_mc['OpenDays'] = (df_mc['Email_Opened'] - df_mc['Email_Sent']).dt.days
df_mc['ClickDays'] = (df_mc['Email_Clicked'] - df_mc['Email_Opened']).dt.days


MarketingComm_end = normalized_cols['MarketingCommunication']['Email_Sent'].max()

marketingComm = normalized_cols['MarketingCommunication'].groupby('CustomerID').agg(
    #Totals of each event
    TotalEmailsSent=('Email_Sent', 'count'),
    TotalEmailsOpened=('Email_Opened', lambda x: x.notna().sum()),
    TotalEmailsClicked=('Email_Clicked', lambda x: x.notna().sum()),
    LastEmailSentDate=('Email_Sent', 'max'),
    LastEmailOpenedDate=('Email_Opened', 'max'),
    LastEmailClickedDate=('Email_Clicked', 'max'),
    AVGOpenDays = ('OpenDays', 'mean'),
    AVGClickDays = ('ClickDays', 'mean'),
    
    AvgOpenDelay_days=('Email_Opened', lambda x: 
                       np.mean((x - normalized_cols['MarketingCommunication'].loc[x.index, 'Email_Sent']).dt.days.dropna()) 
                       if x.notna().any() else np.nan),

    AvgClickDelay_days=('Email_Clicked', lambda x: 
                        np.mean((x - normalized_cols['MarketingCommunication'].loc[x.index, 'Email_Sent']).dt.days.dropna()) 
                        if x.notna().any() else np.nan),


).reset_index()
    

# Engagement ratios
marketingComm['OpenRate'] = marketingComm['TotalEmailsOpened'] / marketingComm['TotalEmailsSent']
marketingComm['ClickRate'] = marketingComm['TotalEmailsClicked'] / marketingComm['TotalEmailsSent']
marketingComm['ClickToOpenRate'] = marketingComm['TotalEmailsClicked'] / marketingComm['TotalEmailsOpened'].replace(0, np.nan)

# Engagement flags
marketingComm['EverOpened'] = (marketingComm['TotalEmailsOpened'] > 0).astype(int)
marketingComm['EverClicked'] = (marketingComm['TotalEmailsClicked'] > 0).astype(int)

# Recency
marketingComm['RecencyLastOpen_days'] = (MarketingComm_end - marketingComm['LastEmailOpenedDate']).dt.days
marketingComm['RecencyLastClick_days'] = (MarketingComm_end - marketingComm['LastEmailClickedDate']).dt.days

In [89]:
marketingComm.head()

,CustomerID,TotalEmailsSent,TotalEmailsOpened,TotalEmailsClicked,LastEmailSentDate,LastEmailOpenedDate,LastEmailClickedDate,AVGOpenDays,AVGClickDays,AvgOpenDelay_days,AvgClickDelay_days,OpenRate,ClickRate,ClickToOpenRate,EverOpened,EverClicked,RecencyLastOpen_days,RecencyLastClick_days
0,1001,8,8,8,2019-10-17,2022-01-12,2022-11-27,818.0,319.0,818.0,1137.0,1.0,1.0,1.0,1,1,352,33
1,1002,9,9,9,2021-08-02,2021-11-20,2022-02-16,110.0,88.0,110.0,198.0,1.0,1.0,1.0,1,1,405,317
2,1003,8,8,8,2021-08-29,2022-07-28,2022-11-22,333.0,117.0,333.0,450.0,1.0,1.0,1.0,1,1,155,38
3,1004,10,10,10,2021-02-03,2021-07-12,2022-09-08,159.0,423.0,159.0,582.0,1.0,1.0,1.0,1,1,536,113
4,1005,7,7,7,2022-03-11,2022-09-20,2022-12-25,193.0,96.0,193.0,289.0,1.0,1.0,1.0,1,1,101,5


In [92]:
cols_with_unique_idrows 

['SubscriptionDetails', 'WebsiteUsage', 'EngagementMetrics', 'Feedback']

In [93]:
SubscriptionDetails = normalized_cols['SubscriptionDetails']
WebsiteUsage = normalized_cols['WebsiteUsage']
EngagementMetrics = normalized_cols['EngagementMetrics']
Feedback = normalized_cols['Feedback']

### Merge all normalized columns

In [94]:
data = pd.read_excel(r"C:\Users\psxea2\Desktop\Amdaritelecomchurn\dataset.xlsx")

In [95]:
# drop columns from the original data that have been normalized
data_reduced = data.drop(columns=cols_to_normalize, axis=1, inplace=False)

In [96]:
data_reduced.columns

Index(['CustomerID', 'Name', 'Age', 'Gender', 'Location', 'Email', 'Phone',
       'Address', 'Segment', 'NPS', 'ChurnLabel', 'Timestamp'],
      dtype='object')

In [103]:
merged_data = data_reduced.copy()

In [104]:
dfs_to_merge = [
                SubscriptionDetails, 
                WebsiteUsage, 
                EngagementMetrics, 
                Feedback, 
                purchaseHistory, 
                paymentHistory, 
                ServiceInteractions, 
                ClickstreamData, 
                marketingComm]

for df in dfs_to_merge:
   merged_data = merged_data.merge(df, on='CustomerID', how='left')

In [105]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12483 entries, 0 to 12482
Data columns (total 73 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CustomerID                  12483 non-null  int64         
 1   Name                        12483 non-null  object        
 2   Age                         12483 non-null  int64         
 3   Gender                      12483 non-null  object        
 4   Location                    12483 non-null  object        
 5   Email                       12483 non-null  object        
 6   Phone                       12483 non-null  object        
 7   Address                     12483 non-null  object        
 8   Segment                     12483 non-null  object        
 9   NPS                         12483 non-null  int64         
 10  ChurnLabel                  12483 non-null  int64         
 11  Timestamp                   12483 non-null  object    

In [ ]:
#Merge the reduced original data with the normalized columns that have unique CustomerID per row

"""
cols_with_unique_idrows = [SubscriptionDetails, WebsiteUsage, EngagementMetrics, Feedback]

for col in cols_with_unique_idrows:
   merged_data = data_reduced.merge(col, on='CustomerID', how='left')

   """

In [ ]:
# Merge the data with the engineered features from the non-unique CustomerID dataframes

"""
engineered_features = [purchaseHistory, paymentHistory, serviceInteractions, ClickstreamData, marketingComm]

for ef in engineered_features:
    merged_data = merged_data.merge(ef, on='CustomerID', how='left')

 """   

In [106]:
merged_data.columns

Index(['CustomerID', 'Name', 'Age', 'Gender', 'Location', 'Email', 'Phone',
       'Address', 'Segment', 'NPS', 'ChurnLabel', 'Timestamp', 'Plan',
       'Start_Date', 'End_Date', 'PageViews', 'TimeSpent(minutes)', 'Logins',
       'Frequency', 'Rating', 'Comment', 'TotalPurchaseFrequency',
       'TotalPurchaseValue', 'ProductList', 'PaymentTypes', 'NumPaymentMethod',
       'AvgLatePayment', 'TotalInteractionType', 'num_calls', 'num_emails',
       'num_chats', 'FirstInteractionDate', 'LastInteractionDate',
       'InteractionDuration_days', 'FirstInteractionType',
       'LastInteractionType', 'Action_count', 'FirstActionTime',
       'LastActionTime', 'AvgTimeBetweenActions_secs', 'TotalDaysActive',
       'MostCommonAction', 'LeastFrequentAction', 'ActivityDuration_days',
       'ActionsPerDay', 'most_recent_action_date', 'TotalPageVisits',
       'unique_pages', 'FirstActionType', 'LastActionType', 'FirstPageVisited',
       'LastPageVisited', 'Recency_days', 'InactivityFlag',
  

In [ ]:

merged_data.shape

In [107]:
merged_data['CustomerID'].nunique()

12483

In [108]:
# any duplicate rows available?
merged_data.duplicated().sum()   

np.int64(0)

In [109]:
# find and display any duplicate rows
merged_data[merged_data.duplicated(keep=False)]

,CustomerID,Name,Age,Gender,Location,Email,Phone,Address,Segment,NPS,...,AVGClickDays,AvgOpenDelay_days,AvgClickDelay_days,OpenRate,ClickRate,ClickToOpenRate,EverOpened,EverClicked,RecencyLastOpen_days,RecencyLastClick_days


In [110]:
merged_data.columns

Index(['CustomerID', 'Name', 'Age', 'Gender', 'Location', 'Email', 'Phone',
       'Address', 'Segment', 'NPS', 'ChurnLabel', 'Timestamp', 'Plan',
       'Start_Date', 'End_Date', 'PageViews', 'TimeSpent(minutes)', 'Logins',
       'Frequency', 'Rating', 'Comment', 'TotalPurchaseFrequency',
       'TotalPurchaseValue', 'ProductList', 'PaymentTypes', 'NumPaymentMethod',
       'AvgLatePayment', 'TotalInteractionType', 'num_calls', 'num_emails',
       'num_chats', 'FirstInteractionDate', 'LastInteractionDate',
       'InteractionDuration_days', 'FirstInteractionType',
       'LastInteractionType', 'Action_count', 'FirstActionTime',
       'LastActionTime', 'AvgTimeBetweenActions_secs', 'TotalDaysActive',
       'MostCommonAction', 'LeastFrequentAction', 'ActivityDuration_days',
       'ActionsPerDay', 'most_recent_action_date', 'TotalPageVisits',
       'unique_pages', 'FirstActionType', 'LastActionType', 'FirstPageVisited',
       'LastPageVisited', 'Recency_days', 'InactivityFlag',
  

### RFM Analysis

In [113]:
current_date = pd.to_datetime(merged_data['LastInteractionDate'].max())

merged_data['recency'] = current_date - merged_data['LastInteractionDate']
merged_data['frequency'] = merged_data['TotalPurchaseFrequency'] * merged_data['Logins']
merged_data['monetary'] = merged_data['TotalPurchaseValue']


r_label = range(5,0, -1)
f_label = range(1, 6)
m_label = range(1, 6)

#segementation according to RFM model
merged_data['recency'] = pd.qcut(merged_data['recency'], q=5,  labels=r_label)
merged_data['frequency'] = pd.qcut(merged_data['frequency'], q=5, labels=r_label)  
merged_data['monetary'] = pd.qcut(merged_data['monetary'], q=5, labels=r_label)

merged_data[['recency', 'frequency', 'monetary']] = merged_data[['recency', 'frequency', 'monetary']].astype('int64')

merged_data['rfm_value'] = merged_data['recency'] + merged_data['frequency'] + merged_data['monetary']




In [114]:
merged_data['rfm_value'].max()

np.int64(15)

In [116]:

bins = [3, 6, 9, 12, 15]
labels = ['at_risk', 'need_attention', 'loyal', 'premium']
merged_data['customer_segments'] = pd.cut(merged_data['rfm_value'], bins=bins, labels=labels)

merged_data.drop(columns=['recency', 'frequency', 'monetary', 'rfm_value'], axis=1, inplace=True)

In [117]:
merged_data.customer_segments.value_counts()

customer_segments
need_attention    4350
loyal             3927
at_risk           2388
premium           1590
Name: count, dtype: int64

### Save cleaned data as a csv file

In [130]:
merged_data.to_csv('cleaned_data.csv', index=False)